In [ ]:
import pandas as pd
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()


In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "core"

albums = pd.read_csv(RAW_DIR / "spotify_albums.csv")
tracks = pd.read_csv(RAW_DIR / "spotify_tracks.csv")

## 1. Quick Inspection

In [ ]:
albums.head()


In [ ]:
albums.info()

In [ ]:
tracks.head()


In [ ]:
tracks.info()

## 2. Check duplicate album names

In [ ]:
albums["album_name"].value_counts().head(20)

In [ ]:
albums[albums["album_name"].duplicated(keep=False)].sort_values("album_name")

## 3. Check duplicate track names

In [ ]:
tracks[tracks["track_name"].duplicated(keep=False)].sort_values("track_name")

## 4. Basic cleaning

In [ ]:
albums_clean = albums.copy()
tracks_clean = tracks.copy()

albums_clean["album_name"] = albums_clean["album_name"].str.strip()
tracks_clean["track_name"] = tracks_clean["track_name"].str.strip()
tracks_clean["album_name"] = tracks_clean["album_name"].str.strip()

albums_clean["release_date"] = pd.to_datetime(
    albums_clean["release_date"],
    errors="coerce"
)
albums_clean["release_year"] = albums_clean["release_date"].dt.year

## 5. Remove exact duplicate rows 

In [ ]:
albums_clean = albums_clean.drop_duplicates()
tracks_clean = tracks_clean.drop_duplicates()

## 6. Before and after summary


In [ ]:
print("Albums:", albums.shape, "->", albums_clean.shape)
print("Tracks:", tracks.shape, "->", tracks_clean.shape)
print("Album rows removed:", len(albums) - len(albums_clean))
print("Track rows removed:", len(tracks) - len(tracks_clean))
print("Repeated track names are preserved across legitimate Spotify releases.")

## 7. Final validation


In [ ]:
print("Clean albums:", albums_clean.shape)
print("Clean tracks:", tracks_clean.shape)

In [ ]:
albums_clean.info()

tracks_clean.info()

In [ ]:
print("Album missing values:")
print(albums_clean.isna().sum())
print("\nTrack missing values:")
print(tracks_clean.isna().sum())

In [ ]:
print("Duplicate album rows:", albums_clean.duplicated().sum())
print("Duplicate track rows:", tracks_clean.duplicated().sum())

In [ ]:
print("Album IDs unique:", albums_clean["album_id"].is_unique)

In [ ]:
print("Track IDs unique:", tracks_clean["track_id"].is_unique)

In [ ]:
albums_clean["album_name"].value_counts().head(20)

In [ ]:
tracks_clean["track_name"].value_counts().head(30)

In [ ]:
tracks_clean["duration_ms"].describe()

In [ ]:
albums_clean["release_date"].min(), albums_clean["release_date"].max()

In [ ]:
albums_clean["album_type"].value_counts()

In [ ]:
tracks_clean["explicit"].value_counts()

In [ ]:
print("Every track belongs to a clean album:", tracks_clean["album_id"].isin(albums_clean["album_id"]).all())

In [ ]:
album_names_match = tracks_clean[["album_id", "album_name"]].drop_duplicates().merge(
    albums_clean[["album_id", "album_name"]],
    on="album_id",
    suffixes=("_track", "_album")
)
print("Track album names match album data:", (album_names_match["album_name_track"] == album_names_match["album_name_album"]).all())

In [ ]:
tracks_clean.groupby("album_id").size().sort_values(ascending=False).head()

In [ ]:
expected = albums_clean.set_index("album_id")["total_tracks"]

actual = tracks_clean.groupby("album_id").size()

comparison = (
    expected.to_frame("expected")
    .join(actual.rename("actual"))
)

comparison["match"] = comparison["expected"] == comparison["actual"]

print("All expected track counts match:", comparison["match"].all())
comparison.loc[~comparison["match"]]

## 8. Export processed data

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

albums_clean.to_csv(PROCESSED_DIR / "spotify_albums_clean.csv", index=False)
tracks_clean.to_csv(PROCESSED_DIR / "spotify_tracks_clean.csv", index=False)

## 9. Verify processed exports

In [ ]:
albums_exported = pd.read_csv(
    PROCESSED_DIR / "spotify_albums_clean.csv",
    parse_dates=["release_date"]
)
tracks_exported = pd.read_csv(PROCESSED_DIR / "spotify_tracks_clean.csv")

print("Exported albums:", albums_exported.shape)
print("Exported tracks:", tracks_exported.shape)
print("Album shape matches:", albums_exported.shape == albums_clean.shape)
print("Track shape matches:", tracks_exported.shape == tracks_clean.shape)
print("Album columns match:", albums_exported.columns.tolist() == albums_clean.columns.tolist())
print("Track columns match:", tracks_exported.columns.tolist() == tracks_clean.columns.tolist())